# BERT Ticket Classifier — Visualization Notebook

This notebook consolidates all key visualizations from the project:
- Class distribution of intent categories
- Baseline vs BERT performance comparison
- Per-class F1 scores (BERT vs baseline)
- Confusion matrices
- Attention weight heatmaps on misclassified examples

**Run the src/ scripts first before executing this notebook.**

Order: `01_preprocessing.py` → `02_baseline.py` → `03_bert_finetune.py` → `04_evaluation.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import json
import os

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

train_df  = pd.read_csv('../data/train.csv')
test_df   = pd.read_csv('../data/test.csv')
label_map = pd.read_csv('../data/label_map.csv')

with open('../outputs/baseline_metrics.json') as f:
    baseline_metrics = json.load(f)
with open('../outputs/bert_metrics.json') as f:
    bert_metrics = json.load(f)

print('Data loaded.')
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Classes: {label_map.shape[0]}")

## 1. Intent Category Distribution

In [ ]:
full_df = pd.concat([train_df, test_df])
label_counts = full_df['label'].value_counts().sort_values(ascending=True)
intent_labels = label_map.set_index('label')['intent']

fig, ax = plt.subplots(figsize=(9, 10))
bars = ax.barh(
    [intent_labels[i] for i in label_counts.index],
    label_counts.values,
    color='#378ADD', alpha=0.85
)
for bar, val in zip(bars, label_counts.values):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

ax.set_xlabel('Number of tickets')
ax.set_title('Intent Category Distribution — Full Dataset', fontweight='bold', pad=12)
ax.grid(axis='x', alpha=0.25, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/intent_distribution.png', bbox_inches='tight')
plt.show()

## 2. Baseline vs BERT — Metric Comparison

In [ ]:
img = mpimg.imread('../outputs/model_comparison.png')
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Baseline Macro F1:  {baseline_metrics['macro_f1']:.4f}")
print(f"BERT Macro F1:      {bert_metrics['macro_f1']:.4f}")
lift = (bert_metrics['macro_f1'] - baseline_metrics['macro_f1']) / baseline_metrics['macro_f1'] * 100
print(f"Relative improvement: +{lift:.1f}%")

## 3. BERT Confusion Matrix

In [ ]:
img = mpimg.imread('../outputs/bert_confusion_matrix.png')
plt.figure(figsize=(14, 12))
plt.imshow(img)
plt.axis('off')
plt.title('BERT — Normalized Confusion Matrix', fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

## 4. Attention Analysis — Misclassified Examples

In [ ]:
img = mpimg.imread('../outputs/attention_analysis.png')
plt.figure(figsize=(14, 10))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

print("""
Interpretation guide:
  - Darker red = higher attention weight from the [CLS] token
  - [CLS] token aggregates context for the final classification decision
  - When the model attends strongly to generic terms ('please', 'help', 'want')
    instead of intent-specific terms ('refund', 'cancel', 'broken'),
    misclassification is more likely
  - These patterns can inform improvements to ticket intake form design
""")